## Goal
Check where Fourier power is added by refinement and whether it is concentrated inside the physical missing wedge.

## Conditions
- input FBP
- baseline: train50/infer50
- under-fill: train50/infer30
- over-mask: train50/infer70

#### für jede difference map

1. Liegt die zusätzliche Power innerhalb der true 50° missing wedge?
2. Bleibt ausserhalb der missing wedge alles ungefähr stabil?
3. Beim 70° case: sieht man zusätzliche Änderungen in der 50°–70° region?
4. Beim 30° case: bleibt die 30°–50° region schwach?

In [3]:
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from ddw.utils.mrctools import load_mrc_data

In [4]:
def load_volume(path):
    vol = load_mrc_data(path)
    if isinstance(vol, torch.Tensor):
        vol = vol.cpu().numpy()
    return vol.astype(np.float32)

In [5]:
def percentile_normalize(img, p_low=1, p_high=99):
    lo, hi = np.percentile(img, [p_low, p_high])
    img = np.clip(img, lo, hi)
    return (img - lo) / (hi - lo + 1e-8)


def get_xy_slice(vol, z_idx=None):
    if z_idx is None:
        z_idx = vol.shape[0] // 2
    return vol[z_idx, :, :]


def get_xz_slice(vol, y_idx=None):
    if y_idx is None:
        y_idx = vol.shape[1] // 2
    return vol[:, y_idx, :]


def get_yz_slice(vol, x_idx=None):
    if x_idx is None:
        x_idx = vol.shape[2] // 2
    return vol[:, :, x_idx]


def compute_fourier_power(vol):

    vol = vol - np.mean(vol) #subtract the mean, bc if the central fourier point is extremly strong, 
    #it can dominate the Power spectrum analysis

    """
    macht aus dem Volumen ein Signal mit Mittelwert ungefähr 0.
    Dadurch vergleichen wir eher die Struktur / Variationen im Tomogramm und nicht den globalen Helligkeits-Offset.
    """
    
    F = np.fft.fftn(vol)
    F = np.fft.fftshift(F)

    power = np.abs(F) ** 2 #Formula for the Power Spectrum
    
    
    return power


def compute_log_power_difference(input_vol, refined_vol):
    #subtract the mean, bc if the central fourier point is extremly strong, 
    #it can dominate the Power spectrum analysis
    input_centered = input_vol - np.mean(input_vol)
    refined_centered = refined_vol - np.mean(refined_vol)
    
    """
    macht aus dem Volumen ein Signal mit Mittelwert ungefähr 0.
    Dadurch vergleichen wir eher die Struktur / Variationen im Tomogramm und nicht den globalen Helligkeits-Offset.
    """

    # compute the fourier transform of input and refined tomo
    F_input = np.fft.fftshift(np.fft.fftn(input_centered))
    F_refined = np.fft.fftshift(np.fft.fftn(refined_centered))

    # compute the fourier power, easier than just normal fft, since there we have complex fourier values, gets complicated
    power_input = np.log1p(np.abs(F_input)**2)
    power_refined = np.log1p(np.abs(F_refined)**2)

    # return the difference, still a 3D volume with the difference everywhere
    return power_refined - power_input